<a href="https://colab.research.google.com/github/PalQuisp/Sis420-int/blob/main/lab04_ori.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# utilizado para la manipulación de directorios y rutas
import os # 20x20 = 400 X, 401 carateres Tita , y=10

# Cálculo científico y vectorial para python
import numpy as np

# Libreria para graficos
from matplotlib import pyplot

# Modulo de optimizacion en scipy
from scipy import optimize


# modulo para cargar archivos en formato MATLAB
# from scipy.io import loadmat

# le dice a matplotlib que incruste gráficos en el cuaderno
%matplotlib inline

In [3]:
import pandas as pd

# La entrada es de 14 elemento contando con x0
input_layer_size  = 14

num_labels = 3


data = pd.read_csv('/content/drive/MyDrive/machine learning/datasets/lab004.csv')

# 2. Separar la variable objetivo 'y' (última columna del dataset)
y_raw = data.iloc[:, -1].values

# Mapear 'y' a índices numéricos de 0 a K-1 automáticamente
uniques, y = np.unique(y_raw, return_inverse=True)
num_labels = len(uniques)

# 3. Separar las características 'X' (todas las columnas excepto la última)
X_raw = data.iloc[:, :-1]

# Convertir variables de texto/categóricas a numéricas si existen
X_df = pd.get_dummies(X_raw, drop_first=True)

# Convertir a matriz NumPy de tipo flotante
X = X_df.values.astype(float)

# 4. Parámetros del modelo
m, n = X.shape
input_layer_size = n

#m = y.size


In [4]:
print(X[0,:])
print(y)

[0.00000000e+00 3.86670000e+04 1.88300000e+03 3.20115980e+01
 9.00000000e+00 5.00000000e+00 3.00000000e+00 3.00000000e+00
 2.81148000e-01 1.56193000e-01 4.37341000e-01 5.55556000e-01
 2.96000000e+02 3.20000000e+01 4.00000000e+01 1.68000000e+02
 3.20000000e+01 4.00000000e+01 0.00000000e+00 2.00000000e+00
 1.00000000e+00 3.00000000e+00 3.00000000e+00 1.30000000e+01
 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 3.30000000e+01 7.60000000e+01 8.44444400e+00
 1.31159360e+01 0.00000000e+00 2.30000000e+01 3.20000000e+01
 6.40000000e+00 9.55510300e+00 0.00000000e+00 3.30000000e+01
 1.08000000e+02 7.71428600e+00 1.16184770e+01 7.61985779e+02
 2.97291830e+07 3.20115979e+07 4.00144973e+06 1.04030736e+07
 4.43887711e+03 1.51169395e+06 2.02639103e+06 5.06597757e+05
 6.80406147e+05 7.61985779e+02 2.97291830e+07 3.20115979e+07
 2.46243061e+06 8.19974671e+06 3.37377700e+00 3.00000000e+00
 1.66666700e+00 2.53333330e+01 1.06666670e+01 0.00000000e+00
 0.00000000e+00 0.000000

In [5]:
def featureNormalize(X):
    X_norm = X.copy()
    mu = np.mean(X, axis=0)
    sigma = np.std(X, axis=0)

    # Evita división por cero si una columna tiene desviación estándar igual a 0
    sigma[sigma == 0] = 1.0

    X_norm = (X - mu) / sigma
    return X_norm, mu, sigma


In [6]:
# llama featureNormalize con los datos cargados
X_norm, mu, sigma = featureNormalize(X)

In [8]:
print(X_norm[0,:])
print(y)

[-1.21590040e+00  2.11202234e-01  1.65265784e-01  2.16930567e-01
  3.01353487e-01  9.35996923e-02  7.78595634e-02  6.74971182e-02
 -9.48870460e-01 -9.48656484e-01 -9.48766736e-01 -8.85604640e-01
  6.16009802e-01  2.28514457e+00  2.67656307e+00  1.18041452e-01
  1.78855957e+00  2.29286366e+00 -2.43374204e-01  2.21214508e+00
  4.65784773e-01  6.70275387e-01  4.33857982e-01  2.47805476e-01
 -1.28698555e-01  0.00000000e+00 -2.25374818e-02 -2.21955787e-02
 -2.12604029e+00 -7.23410375e-01 -3.01882998e-02 -1.99748634e+00
  1.11189545e-01 -1.94661151e-01 -1.27126869e-01 -1.13332659e-02
 -1.48375400e-01 -1.17754284e-01 -3.82115712e-01 -5.29304421e-01
 -1.45817592e-02 -1.13655431e+00 -8.70352915e-01 -6.58881124e-03
  3.03472464e+00  2.17219782e-01  1.98596519e+00  3.06587518e+00
  2.97829179e-03  2.57417935e-01  2.70249088e-03  3.81744230e-01
  2.98594098e-01 -1.38358834e-02  3.02730214e+00  2.16922343e-01
  2.65521095e+00  3.09216394e+00 -9.15209679e-01  5.33056260e-01
  5.47573961e-02 -2.59140

In [ ]:
# Configurar la matriz adecuadamente, y agregar una columna de unos que corresponde al termino de intercepción.
m, n = X.shape
# Agraga el termino de intercepción a A
# X = np.concatenate([np.ones((m, 1)), X_norm], axis=1)
X = X_norm
# X = np.concatenate([np.ones((m, 1)), X], axis=1)

### 1.3 Vectorización de regresión logística

Se utilizará múltiples modelos de regresión logística uno contra todos para construir un clasificador de clases múltiples. Dado que hay 10 clases, deberá entrenar 10 clasificadores de regresión logística separados. Para que esta capacitación sea eficiente, es importante asegurarse de que el código esté bien vectorizado.

En esta sección, se implementará una versión vectorizada de regresión logística que no emplea ningún bucle "for".

Para probar la regresión logística vectorizada, se usara datos personalizados como se definen a continuación.

<a id="section1"></a>
#### 1.3.1 Vectorización de la funcion de costo

Se inicia escribiendo una versión vectorizada de la función de costo. En la regresión logística (no regularizada), la función de costo es

$$ J(\theta) = \frac{1}{m} \sum_{i=1}^m \left[ -y^{(i)} \log \left( h_\theta\left( x^{(i)} \right) \right) - \left(1 - y^{(i)} \right) \log \left(1 - h_\theta \left( x^{(i)} \right) \right) \right] $$

Para calcular cada elemento en la suma, tenemos que calcular $h_\theta(x^{(i)})$ para cada ejemplo $i$, donde $h_\theta(x^{(i)}) = g(\theta^T x^{(i)})$ y $g(z) = \frac{1}{1+e^{-z}}$ es la funcion sigmoidea. Resulta que podemos calcular esto rápidamente para todos los ejemplos usando la multiplicación de matrices. Definamos $X$ y $\theta$ como

$$ X = \begin{bmatrix} - \left( x^{(1)} \right)^T - \\ - \left( x^{(2)} \right)^T - \\ \vdots \\ - \left( x^{(m)} \right)^T - \end{bmatrix} \qquad \text{and} \qquad \theta = \begin{bmatrix} \theta_0 \\ \theta_1 \\ \vdots \\ \theta_n \end{bmatrix} $$

Luego, de calcular el producto matricial $X\theta$, se tiene:

$$ X\theta = \begin{bmatrix} - \left( x^{(1)} \right)^T\theta - \\ - \left( x^{(2)} \right)^T\theta - \\ \vdots \\ - \left( x^{(m)} \right)^T\theta - \end{bmatrix} = \begin{bmatrix} - \theta^T x^{(1)}  - \\ - \theta^T x^{(2)} - \\ \vdots \\ - \theta^T x^{(m)}  - \end{bmatrix} $$

En la última igualdad, usamos el hecho de que $a^Tb = b^Ta$ if $a$ y $b$ son vectores. Esto permite calcular los productos $\theta^T x^{(i)}$ para todos los ejemplos $i$ en una linea de codigo.

#### 1.3.2 Vectorización del gradiente

Recordemos que el gradiente del costo de regresión logística (no regularizado) es un vector donde el elemento $j^{th}$ se define como
$$ \frac{\partial J }{\partial \theta_j} = \frac{1}{m} \sum_{i=1}^m \left( \left( h_\theta\left(x^{(i)}\right) - y^{(i)} \right)x_j^{(i)} \right) $$

Para vectorizar esta operación sobre el conjunto de datos, se inicia escribiendo todas las derivadas parciales explícitamente para todos $\theta_j$,

$$
\begin{align*}
\begin{bmatrix}
\frac{\partial J}{\partial \theta_0} \\
\frac{\partial J}{\partial \theta_1} \\
\frac{\partial J}{\partial \theta_2} \\
\vdots \\
\frac{\partial J}{\partial \theta_n}
\end{bmatrix} = &
\frac{1}{m} \begin{bmatrix}
\sum_{i=1}^m \left( \left(h_\theta\left(x^{(i)}\right) - y^{(i)} \right)x_0^{(i)}\right) \\
\sum_{i=1}^m \left( \left(h_\theta\left(x^{(i)}\right) - y^{(i)} \right)x_1^{(i)}\right) \\
\sum_{i=1}^m \left( \left(h_\theta\left(x^{(i)}\right) - y^{(i)} \right)x_2^{(i)}\right) \\
\vdots \\
\sum_{i=1}^m \left( \left(h_\theta\left(x^{(i)}\right) - y^{(i)} \right)x_n^{(i)}\right) \\
\end{bmatrix} \\
= & \frac{1}{m} \sum_{i=1}^m \left( \left(h_\theta\left(x^{(i)}\right) - y^{(i)} \right)x^{(i)}\right) \\
= & \frac{1}{m} X^T \left( h_\theta(x) - y\right)
\end{align*}
$$

donde

$$  h_\theta(x) - y =
\begin{bmatrix}
h_\theta\left(x^{(1)}\right) - y^{(1)} \\
h_\theta\left(x^{(2)}\right) - y^{(2)} \\
\vdots \\
h_\theta\left(x^{(m)}\right) - y^{(m)}
\end{bmatrix} $$

Nota $x^{(i)}$ es un vector, mientras $h_\theta\left(x^{(i)}\right) - y^{(i)}$ es un escalar(simple número).
Para comprender el último paso de la derivación, dejemos $\beta_i = (h_\theta\left(x^{(m)}\right) - y^{(m)})$ y
observar que:

$$ \sum_i \beta_ix^{(i)} = \begin{bmatrix}
| & | & & | \\
x^{(1)} & x^{(2)} & \cdots & x^{(m)} \\
| & | & & |
\end{bmatrix}
\begin{bmatrix}
\beta_1 \\
\beta_2 \\
\vdots \\
\beta_m
\end{bmatrix} = x^T \beta
$$

donde los valores $\beta_i = \left( h_\theta(x^{(i)} - y^{(i)} \right)$.

La expresión anterior nos permite calcular todas las derivadas parciales sin bucles.
Si se siente cómodo con el álgebra lineal, le recomendamos que trabaje con las multiplicaciones de matrices anteriores para convencerse de que la versión vectorizada hace los mismos cálculos.

<div class="alert alert-box alert-warning">
** Consejo de depuración: ** El código de vectorización a veces puede ser complicado. Una estrategia común para la depuración es imprimir los tamaños de las matrices con las que está trabajando usando la propiedad `shape` de las matrices` numpy`.

Por ejemplo, dada una matriz de datos $X$ de tamaño $100\veces 20$ (100 ejemplos, 20 características) y $\theta$, un vector con tamaño $20$, puede observar que `np.dot (X, theta) `es una operación de multiplicación válida, mientras que` np.dot (theta, X) `no lo es.

Además, si tiene una versión no vectorizada de su código, puede comparar la salida de su código vectorizado y el código no vectorizado para asegurarse de que produzcan las mismas salidas.</div>
<a id="lrCostFunction"></a>

In [9]:
def sigmoid(z):
    """
    Calcula la sigmoide de z.
    """
    return 1.0 / (1.0 + np.exp(-z))

In [14]:

def calcularCosto(theta, X, y):
    # Inicializar algunos valores útiles
    m = y.size  # número de ejemplos de entrenamiento

    # Inicializar la variable del costo
    J = 0

    # Calcular la hipótesis h_theta(x)
    h = sigmoid(X.dot(theta.T))

    # Calcular la función de costo
    J = (1 / m) * np.sum(-y.dot(np.log(h + 1e-15)) - (1 - y).dot(np.log(1 - h + 1e-15)))

    # Retorna el valor escalar del costo J
    return J

In [16]:
def descensoGradiente(X, y, theta, alpha, num_iters):
    # Inicializar algunos valores útiles
    m = y.size  # número de ejemplos de entrenamiento

    # Realizar una copia de theta para no modificar el arreglo original
    theta = theta.copy()

    # Crear una lista vacía para almacenar el historial de costos en cada iteración
    J_history = []

    # Bucle principal de optimización
    for i in range(num_iters):
        # Calcular la hipótesis h_theta(x)
        h = sigmoid(X.dot(theta.T))

        # Actualizar cada parámetro theta restando el gradiente multiplicado por el alpha
        theta = theta - (alpha / m) * (h - y).dot(X)

        # Guardar el costo calculado en la iteración actual para verificar la convergencia
        J_history.append(calcularCosto(theta, X, y))

    # Retorna los parámetros optimizados theta y el historial de costo J_history
    return theta, J_history

In [17]:
def lrCostFunction(theta, X, y, lambda_):
    # Obtener el número de ejemplos de entrenamiento
    m = y.size

    # Convertir etiquetas booleanas a enteros (1 o 0) si es necesario
    if y.dtype == bool:
        y = y.astype(int)

    # Inicializar la variable escalar del costo
    J = 0

    # Inicializar el vector de gradientes con la misma forma de theta
    grad = np.zeros(theta.shape)

    # Calcular la hipótesis h_theta(x) = sigmoid(X * theta)
    h = sigmoid(X.dot(theta))

    # Crear una copia del vector theta para la regularización
    temp = theta.copy()

    # Asignar 0 al término de sesgo (theta_0) para excluirlo de la penalización
    temp[0] = 0

    # Calcular la función de costo regularizada J(theta)
    # Se agrega 1e-15 dentro del logaritmo para prevenir errores numéricos de log(0)
    J = (1 / m) * np.sum(-y * np.log(h + 1e-15) - (1 - y) * np.log(1 - h + 1e-15)) + (lambda_ / (2 * m)) * np.sum(temp ** 2)

    # Calcular el vector de gradientes dJ/dtheta regularizado
    grad = (1 / m) * X.T.dot(h - y) + (lambda_ / m) * temp

    # Retornar el costo escalar J y el vector de gradientes
    return J, grad

#### 1.3.3 Vectorización regularizada de la regresión logística

Una vez implementada la vectorización para la regresión logística, corresponde agregarar regularización a la función de costo.
Para la regresión logística regularizada, la función de costo se define como

$$ J(\theta) = \frac{1}{m} \sum_{i=1}^m \left[ -y^{(i)} \log \left(h_\theta\left(x^{(i)} \right)\right) - \left( 1 - y^{(i)} \right) \log\left(1 - h_\theta \left(x^{(i)} \right) \right) \right] + \frac{\lambda}{2m} \sum_{j=1}^n \theta_j^2 $$

Tomar en cuenta que no debería regularizarse $\theta_0$ que se usa para el término de sesgo. En consecuencia, la derivada parcial del costo de regresión logística regularizado para $\theta_j$ se define como

$$
\begin{align*}
& \frac{\partial J(\theta)}{\partial \theta_0} = \frac{1}{m} \sum_{i=1}^m \left( h_\theta\left( x^{(i)} \right) - y^{(i)} \right) x_j^{(i)}  & \text{for } j = 0 \\
& \frac{\partial J(\theta)}{\partial \theta_0} = \left( \frac{1}{m} \sum_{i=1}^m \left( h_\theta\left( x^{(i)} \right) - y^{(i)} \right) x_j^{(i)} \right) + \frac{\lambda}{m} \theta_j & \text{for } j  \ge 1
\end{align*}
$$

<div class="alert alert-box alert-warning">
** Python/numpy Consejo: ** Al implementar la vectorización para la regresión logística regularizada, a menudo es posible que solo desee sumar y actualizar ciertos elementos de $\theta$. En `numpy`, puede indexar en las matrices para acceder y actualizar solo ciertos elementos.

Por ejemplo, A [:, 3: 5] = B [:, 1: 3] reemplazará las columnas con índice 3 a 5 de A con las columnas con índice 1 a 3 de B.   
Para seleccionar columnas (o filas) hasta el final de la matriz, puede dejar el lado derecho de los dos puntos en blanco.
Por ejemplo, A [:, 2:] solo devolverá elementos desde $3^{rd}$ a las últimas columnas de $A$.Si deja el tamaño de la mano izquierda de los dos puntos en blanco, seleccionará los elementos del principio de la matriz.
Por ejemplo, A [:,: 2] selecciona las dos primeras columnas y es equivalente a A [:, 0: 2]. Además, puede utilizar índices negativos para indexar matrices desde el final.
Por lo tanto, A [:,: -1] selecciona todas las columnas de A excepto la última columna, y A [:, -5:] selecciona la columna $5^{th}$ desde el final hasta la última columna.

Por lo tanto, podría usar esto junto con las operaciones de suma y potencia ($^{**}$) para calcular la suma de solo los elementos que le interesan (por ejemplo, `np.sum (z[1:]**2)`).
</div>


<a id="section2"></a>
### 1.4 Clasificacion One-vs-all
En esta parte del ejercicio, se implementará la clasificación de uno contra todos mediante el entrenamiento de múltiples clasificadores de regresión logística regularizados, uno para cada una de las clases $K$ en nuestro conjunto de datos. En el conjunto de datos de dígitos escritos a mano, $K = 10$, pero su código debería funcionar para cualquier valor de $K$.

El argumento `y` de esta función es un vector de etiquetas de 0 a 9. Al entrenar el clasificador para la clase $k \in \{0, ..., K-1 \} $, querrá un vector K-dimensional de etiquetas $y$, donde $y_j \ in 0, 1$ indica si la instancia de entrenamiento $j ^ {th}$ pertenece a la clase $k$ $(y_j = 1)$, o si pertenece a una clase diferente $(y_j = 0)$.

Además, se utiliza `optimize.minimize` de scipy para este ejercicio.
<a id="oneVsAll"></a>

In [19]:
def OneVsAll(X, y, num_labels, lambda_): # num_labels son las etiquetas o clases de y
    alpha = 0.001 # coeficiente recomendado
    num_iters = 100000 # iteraciones

    m, n = X.shape # numero de ejemplos y numero de columnas
    all_theta = np.zeros((num_labels, n + 1)) # matriz para almacenar theta de cada clase c

    # Agrega unos a la matriz X (columna de bias)
    X = np.concatenate([np.ones((m, 1)), X], axis=1)

    # Iterar para entrenar cada clasificador binario c
    for c in np.arange(num_labels):
        initial_theta = np.zeros(n + 1) # se crea un theta inicial en ceros por cada bucle

        y_actual = np.where(y == c, 1, 0) # si y == c le da valor 1, al resto 0

        # Ejecutar el descenso por el gradiente para la clase c
        theta, J_history = descensoGradiente(initial_theta, X, y_actual, alpha, num_iters)

        # Guardar el vector theta optimizado en la fila c
        all_theta[c] = theta

        # Grafica la convergencia del costo para la clase c
        pyplot.plot(np.arange(len(J_history)), J_history, lw=2, label=f'Clase {c}')
        pyplot.xlabel('Numero de iteraciones')
        pyplot.ylabel('Costo J')
        pyplot.title('Convergencia del Costo por Clase')
        pyplot.legend()

    # Retorna la matriz con todos los theta aprendidos
    return all_theta

In [23]:
def OneVsAllOM(X, y, num_labels, lambda_):
    # Obtener el número de ejemplos (m) y el número de características (n)
    m, n = X.shape

    # Inicializar la matriz all_theta con ceros de tamaño (num_labels x n + 1)
    all_theta = np.zeros((num_labels, n + 1))

    # Agregar la columna de unos a la matriz X para el término de sesgo (bias)
    X = np.concatenate([np.ones((m, 1)), X], axis=1)

    # Iterar sobre cada clase c para entrenar su clasificador correspondiente
    for c in range(num_labels):
        # Inicializar el vector theta en ceros para la clase actual (tamaño n + 1)
        initial_theta = np.zeros(n + 1)

        # Configurar las opciones para el algoritmo de optimización (máximo de iteraciones)
        options = {'maxiter': 50}

        # Crear el vector binario para la clase actual (1 si y == c, 0 en otro caso)
        y_actual = np.where(y == c, 1, 0)

        # Minimizar la función lrCostFunction para la clase actual c
        res = optimize.minimize(
            lrCostFunction,
            initial_theta,
            args=(X, y_actual, lambda_),
            jac=True,
            method='TNC',
            options=options
        )

        # Almacenar los parámetros theta optimizados en la fila de la clase c
        all_theta[c, :] = res.x

    # Retornar la matriz completa de parámetros entrenados
    return all_theta

In [25]:
# Definir el valor del parámetro de regularización lambda
lambda_ = 0.1

# Entrenar todos los clasificadores mediante optimización avanzada
all_theta = OneVsAllOM(X, y, num_labels, lambda_)

# Mostrar el tamaño de la matriz all_theta obtenida
print("Entrenamiento con OneVsAllOM completado.")

/tmp/ipykernel_1860/895411955.py:23: OptimizeWarning: Unknown solver options: maxiter
  res = optimize.minimize(
/tmp/ipykernel_1860/1889279066.py:5: RuntimeWarning: overflow encountered in exp
  return 1.0 / (1.0 + np.exp(-z))


Entrenamiento con OneVsAllOM completado.


In [26]:
print(all_theta) # revisar porque el X0 es el X3

[[-5.17890082e-06 -1.24008471e-04  2.14008036e-06 ... -2.40986061e-08
  -4.49233664e-07  1.19135767e-06]
 [-3.17407943e-06 -4.76256553e-04 -1.05730268e-04 ... -4.01608692e-11
  -6.54828244e-09 -3.88057533e-07]
 [-4.45139782e-03  3.29529321e-04 -8.51994519e-05 ... -1.78256440e-11
   6.69065805e-10  6.65523555e-06]
 ...
 [-1.77563588e-02  7.13486776e-04 -7.11444739e-05 ... -1.55989377e-06
   9.57717788e-10  1.00468193e-07]
 [-2.26907808e-06 -1.02976862e-04  1.10653443e-05 ... -1.08612432e-08
  -1.21944676e-07 -1.14159008e-06]
 [-6.00286770e-06 -3.18556115e-03 -3.13711588e-05 ... -1.42725451e-10
  -1.22238012e-08  8.44611994e-07]]


<a id="section3"></a>
#### 1.4.1 Prediccion One-vs-all

Después de entrenar el clasificador de one-vs-all, se puede usarlo para predecir el dígito contenido en una imagen determinada. Para cada entrada, debe calcular la "probabilidad" de que pertenezca a cada clase utilizando los clasificadores de regresión logística entrenados. La función de predicción one-vs-all seleccionará la clase para la cual el clasificador de regresión logística correspondiente genera la probabilidad más alta y devolverá la etiqueta de clase (0, 1, ..., K-1) como la predicción para el ejemplo de entrada.

In [28]:
def predictOneVsAll(all_theta, X):
    # Obtener el número de ejemplos (filas) de X
    m = X.shape[0]

    # Obtener el número de clases (filas de la matriz all_theta)
    num_labels = all_theta.shape[0]

    # Inicializar el vector de predicciones p con ceros
    p = np.zeros(m, dtype=int)

    # Agregar la columna de unos a la matriz X para el término de sesgo (bias)
    X = np.concatenate([np.ones((m, 1)), X], axis=1)

    # Calcular la matriz de probabilidades aplicando la función sigmoide sobre el producto matricial X * all_theta.T
    h = sigmoid(X.dot(all_theta.T))

    # Obtener el índice de la columna que contiene la máxima probabilidad para cada fila (ejemplo)
    p = np.argmax(h, axis=1)

    # Retornar el vector con las etiquetas predichas
    return p

Una vez que haya terminado, se llama a la función `predictOneVsAll` usando el valor aprendido de $\theta$. Debería apreciarse que la precisión del conjunto de entrenamiento es de aproximadamente 95,1% (es decir, clasifica correctamente el 95,1% de los ejemplos del conjunto de entrenamiento).

In [30]:
# Imprime las dimensiones de la matriz de entrenamiento X
print(X.shape)

# Realiza la prediccion global sobre todo el conjunto de datos X
pred = predictOneVsAll(all_theta, X)

# Calcula y muestra la precision del modelo
print('Precision del conjunto de entrenamiento: {:.2f}%'.format(np.mean(pred == y) * 100))

# Extrae una muestra de prueba (slice de la fila 100 a la 145)
XPrueba = X[100:145, :].copy()

# Muestra la forma original de la muestra (filas, columnas)
print(XPrueba.shape)

# OPCION A (Recomendada): Usar la funcion predictOneVsAll directa sin concatenar manualmente
p = predictOneVsAll(all_theta, XPrueba)

# Imprime las etiquetas predichas
print(p)

# Imprime las etiquetas reales para comparar visualmente
print(y[100:145])

(123117, 93)
Precision del conjunto de entrenamiento: 97.66%
(45, 93)
[3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3
 3 3 3 3 3 3 3 3]
[3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3
 3 3 3 3 3 3 3 3]


/tmp/ipykernel_1860/1889279066.py:5: RuntimeWarning: overflow encountered in exp
  return 1.0 / (1.0 + np.exp(-z))
